In [ ]:
import warnings
warnings.filterwarnings("ignore", category=DeprecationWarning)

# Stock Price Prediction with 5 Models

Notebook này hiện thực pipeline dự báo giá cổ phiếu dựa trên paper, đồng thời so sánh 5 mô hình:

- Linear Regression
- DNN
- CNN
- LSTM
- Hybrid LSTM-GCN

Dữ liệu được tải từ Yahoo Finance, xử lý theo common trading dates, scale bằng Min-Max, xây graph bằng Pearson correlation + Apriori, và đánh giá bằng expanding window backtest.

In [ ]:
!pip -q install yfinance mlxtend

## 1. Import libraries and define global configuration

Phần này import toàn bộ thư viện cần dùng và khai báo các tham số chính cho toàn bộ notebook.

In [ ]:
import os
import math
import random
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error
from sklearn.linear_model import LinearRegression

from mlxtend.frequent_patterns import apriori, association_rules

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

import yfinance as yf

In [ ]:
# =========================
# CONFIG CHUNG
# =========================

SEED = 42

TICKERS = [
    "AAPL", "MSFT", "CMCSA", "COST", "QCOM",
    "ADBE", "SBUX", "INTU", "AMD", "INTC"
]

START_DATE = "2005-01-01"
END_DATE   = "2023-12-31"

# Bám source code tác giả hơn cho HYBRID:
# - baseline chỉ dùng Close
# - hybrid dùng node features = [Close, MA20]
FEATURE_COLS = ["Close", "MA20"]
MA_WINDOW = 20

# Các tham số bám paper / source code tác giả
LOOKBACK_CANDIDATES = [11, 21]
CORR_THRESHOLD = 0.7
LIFT_THRESHOLD = 1.7
DROPOUT = 0.5

APRIORI_MIN_SUPPORT = 0.10
APRIORI_MIN_CONFIDENCE = 0.10
APRIORI_MOVE_THRESHOLD = 0.01

LEARNING_RATE = 0.005
EPOCHS = 40
BATCH_SIZE = 11
PATIENCE = 5

# Nếu True: train window bám source code tác giả hơn
# (gần như toàn bộ lịch sử trước 50 ngày cuối)
USE_REPO_SOURCE_WINDOW = True

# Expanding window
INITIAL_TRAIN_DAYS = 504
TEST_DAYS = 50

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("DEVICE:", DEVICE)


In [ ]:
def set_seed(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed(SEED)

## 2. Data downloading and basic cleaning

Phần này định nghĩa các hàm tải dữ liệu từ Yahoo Finance và chuẩn hóa dữ liệu về cùng một schema để xử lý thuận tiện hơn.

In [ ]:
def flatten_single_ticker_yfinance(df: pd.DataFrame, ticker: str) -> pd.DataFrame:
    """
    Chuẩn hóa output của yfinance về schema:
    Date, Open, High, Low, Close, Adj Close, Volume, Ticker
    """
    temp = df.copy()

    if isinstance(temp.columns, pd.MultiIndex):
        temp.columns = temp.columns.get_level_values(0)

    temp = temp.rename_axis("Date").reset_index()
    temp.columns = [str(col).strip() for col in temp.columns]

    rename_map = {
        "AdjClose": "Adj Close",
        "Adj close": "Adj Close",
        "adj close": "Adj Close",
    }
    temp = temp.rename(columns=rename_map)

    if "Adj Close" not in temp.columns and "Close" in temp.columns:
        temp["Adj Close"] = temp["Close"]

    temp["Ticker"] = ticker

    expected_cols = ["Date", "Open", "High", "Low", "Close", "Adj Close", "Volume", "Ticker"]
    existing_cols = [col for col in expected_cols if col in temp.columns]
    temp = temp[existing_cols].copy()

    return temp

In [ ]:
def download_yfinance_data(tickers, start_date=START_DATE, end_date=END_DATE):
    """
    Tải dữ liệu từng ticker riêng để tránh lỗi format cột.
    """
    all_frames = []

    for ticker in tickers:
        print(f"Downloading data for {ticker} ...")
        df = yf.download(
            ticker,
            start=start_date,
            end=end_date,
            auto_adjust=False,
            progress=False
        )
        df = flatten_single_ticker_yfinance(df, ticker)
        all_frames.append(df)

    raw_df = pd.concat(all_frames, axis=0, ignore_index=True)
    raw_df["Date"] = pd.to_datetime(raw_df["Date"])
    raw_df = raw_df.sort_values(["Date", "Ticker"]).reset_index(drop=True)

    return raw_df

## 3. Download raw stock data

Thực hiện tải dữ liệu thật cho 10 mã cổ phiếu đã chọn.

In [ ]:
raw_df = download_yfinance_data(TICKERS, START_DATE, END_DATE)

all_dates = sorted(raw_df["Date"].unique())
if USE_REPO_SOURCE_WINDOW:
    INITIAL_TRAIN_DAYS = len(all_dates) - TEST_DAYS - 1

print("Raw data shape:", raw_df.shape)
print("Unique trading dates:", len(all_dates))
print("INITIAL_TRAIN_DAYS:", INITIAL_TRAIN_DAYS)
display(raw_df.head())
print("\nColumns:", raw_df.columns.tolist())


## 4. Basic data inspection

Kiểm tra số lượng dòng theo từng mã, số lượng giá trị thiếu và làm sạch dữ liệu cơ bản.

In [ ]:
print("Số dòng mỗi ticker:")
display(raw_df.groupby("Ticker").size().sort_index())

print("\nSố giá trị thiếu mỗi cột:")
display(raw_df.isna().sum())

raw_df = raw_df.dropna().copy()
raw_df = raw_df.sort_values(["Date", "Ticker"]).reset_index(drop=True)

print("\nShape sau khi dropna:", raw_df.shape)

## 5. Exploratory data analysis

Vẽ một số biểu đồ cơ bản như normalized close, moving averages và correlation heatmap để quan sát dữ liệu.

In [ ]:
def get_close_pivot(df, tickers):
    pivot = df.pivot(index="Date", columns="Ticker", values="Close")
    pivot = pivot[tickers].copy()
    pivot = pivot.dropna()
    return pivot

close_pivot = get_close_pivot(raw_df, TICKERS)
print(close_pivot.shape)
display(close_pivot.head())

In [ ]:
sample_ticker = "AAPL"

series = close_pivot[sample_ticker].copy()
norm_series = (series - series.min()) / (series.max() - series.min())
ma50 = norm_series.rolling(50).mean()
ma200 = norm_series.rolling(200).mean()

plt.figure(figsize=(14, 5))
plt.plot(norm_series.index, norm_series.values, label=f"{sample_ticker} Normalized Close")
plt.plot(ma50.index, ma50.values, label="MA50")
plt.plot(ma200.index, ma200.values, label="MA200")
plt.title(f"{sample_ticker} - Normalized Close with MA50 and MA200")
plt.xlabel("Date")
plt.ylabel("Normalized Price")
plt.legend()
plt.show()

In [ ]:
returns_pivot = close_pivot.pct_change().dropna()
corr_matrix = returns_pivot.corr()

plt.figure(figsize=(8, 6))
plt.imshow(corr_matrix.values, aspect="auto")
plt.colorbar()
plt.xticks(range(len(TICKERS)), TICKERS, rotation=45)
plt.yticks(range(len(TICKERS)), TICKERS)
plt.title("Correlation Heatmap of Daily Returns")
plt.show()

## 6. Keep only common trading dates

Để đảm bảo tất cả cổ phiếu có cùng trục thời gian, ta chỉ giữ các ngày giao dịch xuất hiện ở tất cả ticker.

In [ ]:
def get_common_dates(df, tickers):
    date_sets = []
    for ticker in tickers:
        s = set(df.loc[df["Ticker"] == ticker, "Date"].unique())
        date_sets.append(s)

    common_dates = sorted(list(set.intersection(*date_sets)))
    common_dates = pd.to_datetime(common_dates)
    return common_dates

common_dates = get_common_dates(raw_df, TICKERS)
print("Số common dates:", len(common_dates))
print("Từ", common_dates[0], "đến", common_dates[-1])

raw_df = raw_df[raw_df["Date"].isin(common_dates)].copy()
raw_df = raw_df.sort_values(["Date", "Ticker"]).reset_index(drop=True)
print("Shape sau khi giữ common dates:", raw_df.shape)

## 7. Min-Max scaling by ticker

Dữ liệu được scale riêng cho từng ticker bằng Min-Max scaler. Scaler chỉ được fit trên train window để tránh data leakage.

In [ ]:
def fit_minmax_scalers_per_ticker(train_df, tickers, feature_cols):
    scaler_dict = {}

    for ticker in tickers:
        sub = train_df[train_df["Ticker"] == ticker].sort_values("Date").copy()
        scaler = MinMaxScaler()
        scaler.fit(sub[feature_cols].values)
        scaler_dict[ticker] = scaler

    return scaler_dict

In [ ]:
def apply_scalers_per_ticker(df, scaler_dict, tickers, feature_cols):
    parts = []

    for ticker in tickers:
        sub = df[df["Ticker"] == ticker].sort_values("Date").copy()
        scaler = scaler_dict[ticker]
        sub.loc[:, feature_cols] = scaler.transform(sub[feature_cols].values)
        parts.append(sub)

    out = pd.concat(parts, axis=0, ignore_index=True)
    out = out.sort_values(["Date", "Ticker"]).reset_index(drop=True)
    return out

In [ ]:
def scale_window(raw_df, tickers, feature_cols, train_end_date, transform_upto_date):
    """
    Bám source code tác giả hơn:
    - fit MinMax theo từng ticker trên Close của TRAIN
    - transform Close cho toàn bộ window
    - tạo MA20 từ Close đã scale
    """
    train_df = raw_df[raw_df["Date"] <= train_end_date].copy()
    window_df = raw_df[raw_df["Date"] <= transform_upto_date].copy()

    scaler_dict = fit_minmax_scalers_per_ticker(train_df, tickers, ["Close"])
    scaled_window_df = apply_scalers_per_ticker(window_df, scaler_dict, tickers, ["Close"])

    scaled_window_df = scaled_window_df.sort_values(["Ticker", "Date"]).copy()
    scaled_window_df["MA20"] = (
        scaled_window_df.groupby("Ticker")["Close"]
        .transform(lambda s: s.rolling(window=MA_WINDOW, min_periods=MA_WINDOW).mean())
    )

    scaled_window_df = scaled_window_df.dropna(subset=["MA20"]).reset_index(drop=True)
    return scaled_window_df, scaler_dict


## 8. Graph construction using Pearson correlation and Apriori

Graph giữa các cổ phiếu được xây từ hai nguồn quan hệ:
- Pearson correlation trên daily returns
- Apriori association rules

Sau đó hai adjacency matrix được kết hợp để phục vụ mô hình hybrid.

In [ ]:
def build_correlation_adjacency(train_df, tickers, corr_threshold=CORR_THRESHOLD):
    """
    Bám source code tác giả hơn:
    - Pearson trực tiếp trên Close đã scale
    - dùng binary adjacency
    - chỉ giữ tương quan dương > threshold
    """
    close_pivot = train_df.pivot(index="Date", columns="Ticker", values="Close")[tickers]
    corr = close_pivot.corr(method="pearson").fillna(0.0)

    n = len(tickers)
    adj = np.zeros((n, n), dtype=np.float32)

    for i in range(n):
        for j in range(i + 1, n):
            c = float(corr.iloc[i, j])
            if c > corr_threshold:
                adj[i, j] = 1.0
                adj[j, i] = 1.0

    return adj, corr


In [ ]:
def build_apriori_adjacency(
    train_df,
    tickers,
    min_support=APRIORI_MIN_SUPPORT,
    min_confidence=APRIORI_MIN_CONFIDENCE,
    lift_threshold=LIFT_THRESHOLD,
    move_threshold=APRIORI_MOVE_THRESHOLD
):
    """
    Bám source code tác giả hơn:
    - Apriori chạy trên Close đã scale
    - tạo transaction theo ngưỡng 0.01
    - support=0.1, confidence=0.1, lift=1.7
    - graph nhị phân
    """
    close_pivot = train_df.pivot(index="Date", columns="Ticker", values="Close")[tickers]

    transactions = []

    for _, row in close_pivot.iterrows():
        up_data = []
        down_data = []

        for ticker in tickers:
            val = float(row[ticker])

            if val > move_threshold:
                up_data.append(ticker)
            elif val < move_threshold:
                down_data.append(ticker)

        if len(up_data) > 0:
            transactions.append(up_data)
        if len(down_data) > 0:
            transactions.append(down_data)

    n = len(tickers)
    adj = np.zeros((n, n), dtype=np.float32)

    if len(transactions) == 0:
        return adj, None, None

    basket = pd.DataFrame(transactions)
    basket = basket.apply(lambda x: pd.Series(1, index=x.dropna()), axis=1).fillna(0)

    freq_items = apriori(basket, min_support=min_support, use_colnames=True)

    if len(freq_items) == 0:
        return adj, freq_items, None

    rules = association_rules(freq_items, metric="lift", min_threshold=lift_threshold)

    if len(rules) == 0:
        return adj, freq_items, rules

    rules = rules[rules["confidence"] >= min_confidence].copy()

    ticker_to_idx = {ticker: i for i, ticker in enumerate(tickers)}

    for _, row in rules.iterrows():
        antecedents = list(row["antecedents"])
        consequents = list(row["consequents"])

        if len(antecedents) == 1 and len(consequents) == 1:
            a = antecedents[0]
            b = consequents[0]

            if a in ticker_to_idx and b in ticker_to_idx:
                i = ticker_to_idx[a]
                j = ticker_to_idx[b]

                adj[i, j] = 1.0
                adj[j, i] = 1.0

    return adj, freq_items, rules


In [ ]:
def normalize_adjacency(adj):
    A = adj.copy().astype(np.float32)
    np.fill_diagonal(A, 1.0)

    degree = A.sum(axis=1)
    degree_inv_sqrt = np.power(degree + 1e-8, -0.5)
    D_inv_sqrt = np.diag(degree_inv_sqrt)

    A_hat = D_inv_sqrt @ A @ D_inv_sqrt
    return A_hat.astype(np.float32)

In [ ]:
def build_hybrid_graph(
    train_df,
    tickers,
    corr_threshold=CORR_THRESHOLD,
    lift_threshold=LIFT_THRESHOLD,
    min_support=APRIORI_MIN_SUPPORT,
    min_confidence=APRIORI_MIN_CONFIDENCE
):
    corr_adj, corr_matrix = build_correlation_adjacency(
        train_df, tickers, corr_threshold
    )

    apriori_adj, freq_items, rules = build_apriori_adjacency(
        train_df,
        tickers,
        min_support=min_support,
        min_confidence=min_confidence,
        lift_threshold=lift_threshold,
        move_threshold=APRIORI_MOVE_THRESHOLD
    )

    combined_adj = np.maximum(corr_adj, apriori_adj)
    norm_adj = normalize_adjacency(combined_adj)

    return {
        "corr_adj": corr_adj,
        "apriori_adj": apriori_adj,
        "combined_adj": combined_adj,
        "norm_adj": norm_adj,
        "corr_matrix": corr_matrix,
        "freq_items": freq_items,
        "rules": rules
    }


## 9. Build tensors and supervised samples

Phần này biến dữ liệu bảng thành tensor để đưa vào mô hình học máy và học sâu.

In [ ]:
def build_feature_tensors(scaled_df, tickers, feature_cols):
    scaled_df = scaled_df.sort_values(["Date", "Ticker"]).copy()

    close_mat = scaled_df.pivot(index="Date", columns="Ticker", values="Close")[tickers].values

    feature_mats = []
    for col in feature_cols:
        mat = scaled_df.pivot(index="Date", columns="Ticker", values=col)[tickers].values
        feature_mats.append(mat)

    node_feat_tensor = np.stack(feature_mats, axis=0).transpose(1, 2, 0)
    dates = scaled_df["Date"].drop_duplicates().sort_values().tolist()

    return dates, close_mat.astype(np.float32), node_feat_tensor.astype(np.float32)

In [ ]:
def make_samples_from_tensors(close_mat, node_feat_tensor, dates, lookback):
    X_seq = []
    X_node = []
    y = []
    target_dates = []

    T = close_mat.shape[0]

    for t in range(lookback - 1, T - 1):
        seq = close_mat[t - lookback + 1:t + 1]     # [lookback, N]
        seq = seq.T[..., np.newaxis]                # [N, lookback, 1]

        node = node_feat_tensor[t]                  # [N, F]
        target = close_mat[t + 1]                   # [N]

        X_seq.append(seq)
        X_node.append(node)
        y.append(target)
        target_dates.append(dates[t + 1])

    X_seq = np.array(X_seq, dtype=np.float32)
    X_node = np.array(X_node, dtype=np.float32)
    y = np.array(y, dtype=np.float32)

    return X_seq, X_node, y, target_dates


def make_per_stock_close_samples(close_mat, dates, lookback):
    """
    Tạo sample baseline theo từng stock riêng biệt.
    Input:
        close_mat: [T, N]
    Output:
        X_seq: [num_samples, lookback, 1]
        y:     [num_samples, 1]
        meta:  list chứa (target_date, ticker_idx)
    """
    X_seq = []
    y = []
    meta = []

    T, N = close_mat.shape

    for stock_idx in range(N):
        series = close_mat[:, stock_idx]

        for t in range(lookback - 1, T - 1):
            seq = series[t - lookback + 1:t + 1]      # [lookback]
            target = series[t + 1]                    # scalar
            target_date = dates[t + 1]

            X_seq.append(seq[:, np.newaxis])          # [lookback, 1]
            y.append([target])
            meta.append((target_date, stock_idx))

    X_seq = np.array(X_seq, dtype=np.float32)
    y = np.array(y, dtype=np.float32)

    return X_seq, y, meta


def split_train_test_per_stock(X_seq, y, meta, target_date):
    train_idx = [i for i, (d, _) in enumerate(meta) if d < target_date]
    test_idx  = [i for i, (d, _) in enumerate(meta) if d == target_date]

    X_train = X_seq[train_idx]
    y_train = y[train_idx]

    X_test = X_seq[test_idx]
    y_test = y[test_idx]
    meta_test = [meta[i] for i in test_idx]

    return X_train, y_train, X_test, y_test, meta_test


## 10. Dataset utilities

Tạo Dataset cho PyTorch và hàm chuyển sequence data sang dạng tabular để phục vụ Linear Regression và DNN kiểu fully connected.

In [ ]:
class TimeGraphDataset(Dataset):
    def __init__(self, X_seq, X_node, y):
        self.X_seq = torch.tensor(X_seq, dtype=torch.float32)
        self.X_node = torch.tensor(X_node, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.float32)

    def __len__(self):
        return len(self.X_seq)

    def __getitem__(self, idx):
        return self.X_seq[idx], self.X_node[idx], self.y[idx]


class PerStockSequenceDataset(Dataset):
    def __init__(self, X_seq, y):
        self.X_seq = torch.tensor(X_seq, dtype=torch.float32)   # [B, L, 1]
        self.y = torch.tensor(y, dtype=torch.float32)           # [B, 1]

    def __len__(self):
        return len(self.X_seq)

    def __getitem__(self, idx):
        return self.X_seq[idx], self.y[idx]


In [ ]:
def build_tabular_features(X_seq, X_node, use_node_features=True):
    seq_flat = X_seq.squeeze(-1).reshape(len(X_seq), -1)
    if use_node_features:
        node_flat = X_node.reshape(len(X_node), -1)
        X_tab = np.concatenate([seq_flat, node_flat], axis=1)
    else:
        X_tab = seq_flat
    return X_tab.astype(np.float32)

## 11. Model definitions

Notebook này so sánh 5 mô hình:
1. Linear Regression
2. DNN
3. CNN
4. LSTM
5. Hybrid LSTM-GCN

### 11.1 Linear Regression

Linear Regression được huấn luyện trên dữ liệu đã flatten thành vector đặc trưng.

In [ ]:
# Placeholder cell for Linear Regression baseline
# No custom PyTorch architecture is needed because this model uses:
# model = LinearRegression()
# defined and executed inside run_expanding_window_backtest().

### 11.2 DNN baseline

In [ ]:
class DNNModel(nn.Module):
    def __init__(self, lookback, hidden1=128, hidden2=64, dropout=DROPOUT):
        super().__init__()

        self.net = nn.Sequential(
            nn.Linear(lookback, hidden1),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden1, hidden2),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden2, 1)
        )

    def forward(self, x_seq):
        # x_seq: [B, L, 1]
        x = x_seq.squeeze(-1)   # [B, L]
        return self.net(x)


### 11.3 CNN baseline

In [ ]:
class CNN1DModel(nn.Module):
    def __init__(self, conv_channels=32, hidden=64, dropout=DROPOUT):
        super().__init__()

        self.conv1 = nn.Conv1d(in_channels=1, out_channels=conv_channels, kernel_size=3, padding=1)
        self.conv2 = nn.Conv1d(in_channels=conv_channels, out_channels=conv_channels, kernel_size=3, padding=1)
        self.pool = nn.AdaptiveAvgPool1d(1)

        self.head = nn.Sequential(
            nn.Linear(conv_channels, hidden),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden, 1)
        )

    def forward(self, x_seq):
        # x_seq: [B, L, 1]
        x = x_seq.transpose(1, 2)   # [B, 1, L]

        h = F.relu(self.conv1(x))
        h = F.relu(self.conv2(h))
        h = self.pool(h).squeeze(-1)   # [B, conv_channels]

        return self.head(h)


### 11.4 Standalone LSTM

In [ ]:
class LSTMOnlyModel(nn.Module):
    def __init__(
        self,
        input_size=1,
        hidden_size=64,
        num_layers=2,
        dropout=DROPOUT,
        mlp_hidden=32
    ):
        super().__init__()

        self.lstm = nn.LSTM(
            input_size=input_size,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True,
            dropout=dropout if num_layers > 1 else 0.0
        )

        self.dropout = nn.Dropout(dropout)

        self.head = nn.Sequential(
            nn.Linear(hidden_size, mlp_hidden),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(mlp_hidden, 1)
        )

    def forward(self, x_seq):
        # x_seq: [B, L, 1]
        out, _ = self.lstm(x_seq)
        h_last = self.dropout(out[:, -1, :])
        return self.head(h_last)


### 11.5 Hybrid LSTM-GCN

In [ ]:
class GraphConvLayer(nn.Module):
    def __init__(self, in_features, out_features):
        super().__init__()
        self.linear = nn.Linear(in_features, out_features, bias=False)

    def forward(self, x, adj):
        support = self.linear(x)
        out = torch.einsum("ij,bjf->bif", adj, support)
        return out

In [ ]:
class HybridLSTMGCN(nn.Module):
    def __init__(
        self,
        seq_input_size=1,
        node_input_size=len(FEATURE_COLS),
        lstm_hidden=64,
        lstm_layers=2,
        gcn_hidden=32,
        fusion_hidden=64,
        dropout=DROPOUT
    ):
        super().__init__()

        self.lstm = nn.LSTM(
            input_size=seq_input_size,
            hidden_size=lstm_hidden,
            num_layers=lstm_layers,
            batch_first=True,
            dropout=dropout if lstm_layers > 1 else 0.0
        )

        self.gcn1 = GraphConvLayer(node_input_size, gcn_hidden)
        self.gcn2 = GraphConvLayer(gcn_hidden, gcn_hidden)

        self.dropout = nn.Dropout(dropout)

        self.fusion_head = nn.Sequential(
            nn.Linear(lstm_hidden + gcn_hidden, fusion_hidden),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(fusion_hidden, 1)
        )

    def forward(self, x_seq, x_node, adj):
        B, N, L, C = x_seq.shape

        seq_in = x_seq.reshape(B * N, L, C)
        lstm_out, _ = self.lstm(seq_in)
        h_lstm = lstm_out[:, -1, :]
        h_lstm = self.dropout(h_lstm)
        h_lstm = h_lstm.view(B, N, -1)

        h_gnn = self.gcn1(x_node, adj)
        h_gnn = F.relu(h_gnn)
        h_gnn = self.dropout(h_gnn)

        h_gnn = self.gcn2(h_gnn, adj)
        h_gnn = F.relu(h_gnn)
        h_gnn = self.dropout(h_gnn)

        h = torch.cat([h_lstm, h_gnn], dim=-1)
        y_hat = self.fusion_head(h).squeeze(-1)

        return y_hat

## 12. Training utilities

Phần này định nghĩa early stopping, train-validation split theo thời gian, DataLoader và hàm huấn luyện cho các mô hình PyTorch.

In [ ]:
class EarlyStopping:
    def __init__(self, patience=5, min_delta=1e-5):
        self.patience = patience
        self.min_delta = min_delta
        self.best_loss = None
        self.counter = 0
        self.best_state = None

    def step(self, val_loss, model):
        if self.best_loss is None or val_loss < self.best_loss - self.min_delta:
            self.best_loss = val_loss
            self.counter = 0
            self.best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
            return False
        else:
            self.counter += 1
            return self.counter >= self.patience

In [ ]:
def split_train_val_time_order(X_seq, X_node, y, val_ratio=0.1):
    n = len(X_seq)
    n_val = max(1, int(n * val_ratio))
    n_train = n - n_val

    X_seq_train = X_seq[:n_train]
    X_node_train = X_node[:n_train]
    y_train = y[:n_train]

    X_seq_val = X_seq[n_train:]
    X_node_val = X_node[n_train:]
    y_val = y[n_train:]

    return (X_seq_train, X_node_train, y_train), (X_seq_val, X_node_val, y_val)


def split_train_val_time_order_per_stock(X_seq, y, val_ratio=0.1):
    n = len(X_seq)
    n_val = max(1, int(n * val_ratio))
    n_train = n - n_val

    X_train = X_seq[:n_train]
    y_train = y[:n_train]

    X_val = X_seq[n_train:]
    y_val = y[n_train:]

    return (X_train, y_train), (X_val, y_val)


In [ ]:
def make_loaders(
    X_seq_train, X_node_train, y_train,
    X_seq_val, X_node_val, y_val,
    batch_size=BATCH_SIZE
):
    train_ds = TimeGraphDataset(X_seq_train, X_node_train, y_train)
    val_ds = TimeGraphDataset(X_seq_val, X_node_val, y_val)

    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=False)
    val_loader = DataLoader(val_ds, batch_size=batch_size, shuffle=False)

    return train_loader, val_loader


def make_per_stock_loaders(
    X_train, y_train,
    X_val, y_val,
    batch_size=BATCH_SIZE
):
    train_ds = PerStockSequenceDataset(X_train, y_train)
    val_ds = PerStockSequenceDataset(X_val, y_val)

    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=False)
    val_loader = DataLoader(val_ds, batch_size=batch_size, shuffle=False)

    return train_loader, val_loader


In [ ]:
def train_torch_model(
    model,
    train_loader,
    val_loader,
    model_type,
    adj_tensor=None,
    lr=LEARNING_RATE,
    epochs=EPOCHS,
    patience=PATIENCE
):
    model = model.to(DEVICE)
    criterion = nn.MSELoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    early_stopper = EarlyStopping(patience=patience, min_delta=1e-5)

    history = {"train_loss": [], "val_loss": []}

    for epoch in range(epochs):
        model.train()
        train_losses = []

        for x_seq, x_node, y in train_loader:
            x_seq = x_seq.to(DEVICE)
            x_node = x_node.to(DEVICE)
            y = y.to(DEVICE)

            optimizer.zero_grad()

            if model_type == "hybrid":
                pred = model(x_seq, x_node, adj_tensor)
            else:
                raise ValueError(f"Unsupported model_type in train_torch_model: {model_type}")

            loss = criterion(pred, y)
            loss.backward()
            optimizer.step()

            train_losses.append(loss.item())

        avg_train_loss = float(np.mean(train_losses)) if len(train_losses) > 0 else np.nan

        model.eval()
        val_losses = []

        with torch.no_grad():
            for x_seq, x_node, y in val_loader:
                x_seq = x_seq.to(DEVICE)
                x_node = x_node.to(DEVICE)
                y = y.to(DEVICE)

                if model_type == "hybrid":
                    pred = model(x_seq, x_node, adj_tensor)
                else:
                    raise ValueError(f"Unsupported model_type in train_torch_model: {model_type}")

                loss = criterion(pred, y)
                val_losses.append(loss.item())

        avg_val_loss = float(np.mean(val_losses)) if len(val_losses) > 0 else np.nan
        history["train_loss"].append(avg_train_loss)
        history["val_loss"].append(avg_val_loss)

        print(
            f"Epoch {epoch+1:02d}/{epochs} | "
            f"train_loss={avg_train_loss:.6f} | val_loss={avg_val_loss:.6f}"
        )

        should_stop = early_stopper.step(avg_val_loss, model)
        if should_stop:
            print("Early stopping triggered.")
            break

    if early_stopper.best_state is not None:
        model.load_state_dict(early_stopper.best_state)

    return model, history


def train_per_stock_torch_model(
    model,
    train_loader,
    val_loader,
    lr=LEARNING_RATE,
    epochs=EPOCHS,
    patience=PATIENCE
):
    model = model.to(DEVICE)
    criterion = nn.MSELoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    early_stopper = EarlyStopping(patience=patience, min_delta=1e-5)

    history = {"train_loss": [], "val_loss": []}

    for epoch in range(epochs):
        model.train()
        train_losses = []

        for x_seq, y in train_loader:
            x_seq = x_seq.to(DEVICE)
            y = y.to(DEVICE)

            optimizer.zero_grad()
            pred = model(x_seq)
            loss = criterion(pred, y)
            loss.backward()
            optimizer.step()

            train_losses.append(loss.item())

        avg_train_loss = float(np.mean(train_losses)) if len(train_losses) > 0 else np.nan

        model.eval()
        val_losses = []

        with torch.no_grad():
            for x_seq, y in val_loader:
                x_seq = x_seq.to(DEVICE)
                y = y.to(DEVICE)

                pred = model(x_seq)
                loss = criterion(pred, y)
                val_losses.append(loss.item())

        avg_val_loss = float(np.mean(val_losses)) if len(val_losses) > 0 else np.nan
        history["train_loss"].append(avg_train_loss)
        history["val_loss"].append(avg_val_loss)

        print(
            f"Epoch {epoch+1:02d}/{epochs} | "
            f"train_loss={avg_train_loss:.6f} | val_loss={avg_val_loss:.6f}"
        )

        should_stop = early_stopper.step(avg_val_loss, model)
        if should_stop:
            print("Early stopping triggered.")
            break

    if early_stopper.best_state is not None:
        model.load_state_dict(early_stopper.best_state)

    return model, history


## 13. Evaluation metrics and model factory

In [ ]:
def regression_metrics(y_true, y_pred):
    y_true = np.asarray(y_true).reshape(-1)
    y_pred = np.asarray(y_pred).reshape(-1)

    mse = mean_squared_error(y_true, y_pred)
    rmse = math.sqrt(mse)
    mae = mean_absolute_error(y_true, y_pred)

    return {
        "MSE": mse,
        "RMSE": rmse,
        "MAE": mae
    }

In [ ]:
def create_model(model_type, num_stocks, lookback, node_feat_dim):
    if model_type == "dnn":
        return DNNModel(
            lookback=lookback,
            hidden1=128,
            hidden2=64,
            dropout=DROPOUT
        )

    if model_type == "cnn":
        return CNN1DModel(
            conv_channels=32,
            hidden=64,
            dropout=DROPOUT
        )

    if model_type == "lstm":
        return LSTMOnlyModel(
            input_size=1,
            hidden_size=64,
            num_layers=2,
            dropout=DROPOUT,
            mlp_hidden=32
        )

    if model_type == "hybrid":
        return HybridLSTMGCN(
            seq_input_size=1,
            node_input_size=node_feat_dim,
            lstm_hidden=64,
            lstm_layers=2,
            gcn_hidden=32,
            fusion_hidden=64,
            dropout=DROPOUT
        )

    raise ValueError(f"Unsupported model_type: {model_type}")


## 14. Expanding window backtest

Phần quan trọng nhất của notebook: mô phỏng quá trình train-test theo thời gian để đánh giá công bằng các mô hình.

In [ ]:
def run_expanding_window_backtest(
    raw_df,
    tickers,
    feature_cols,
    lookback=11,
    initial_train_days=INITIAL_TRAIN_DAYS,
    test_days=TEST_DAYS,
    model_type="hybrid",
    lr=LEARNING_RATE,
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    patience=PATIENCE
):
    all_dates = sorted(raw_df["Date"].unique())

    assert initial_train_days > lookback + 5, "INITIAL_TRAIN_DAYS quá nhỏ"
    assert initial_train_days + test_days < len(all_dates), "Không đủ số ngày để backtest"

    all_rows = []

    for step in range(test_days):
        print("=" * 90)
        print(f"[{model_type.upper()}] Expanding step {step+1}/{test_days}")

        train_last_idx = initial_train_days - 1 + step
        target_idx = train_last_idx + 1

        train_end_date = all_dates[train_last_idx]
        target_date = all_dates[target_idx]

        print("Train end date :", pd.to_datetime(train_end_date))
        print("Target date    :", pd.to_datetime(target_date))

        scaled_window_df, scaler_dict = scale_window(
            raw_df=raw_df,
            tickers=tickers,
            feature_cols=feature_cols,
            train_end_date=train_end_date,
            transform_upto_date=target_date
        )

        dates, close_mat, node_feat_tensor = build_feature_tensors(
            scaled_window_df,
            tickers,
            feature_cols
        )

        # =========================
        # BASELINES: per-stock, close-only
        # =========================
        if model_type in ["linear", "dnn", "cnn", "lstm"]:
            X_seq_ps, y_ps, meta_ps = make_per_stock_close_samples(
                close_mat=close_mat,
                dates=dates,
                lookback=lookback
            )

            X_train_all, y_train_all, X_test, y_test, meta_test = split_train_test_per_stock(
                X_seq_ps, y_ps, meta_ps, target_date
            )

            if model_type == "linear":
                X_tab_train = X_train_all.squeeze(-1)   # [samples, lookback]
                X_tab_test = X_test.squeeze(-1)

                model = LinearRegression()
                model.fit(X_tab_train, y_train_all.ravel())
                pred = model.predict(X_tab_test).reshape(-1, 1)

            else:
                (X_train, y_train), (X_val, y_val) = split_train_val_time_order_per_stock(
                    X_train_all, y_train_all, val_ratio=0.10
                )

                train_loader, val_loader = make_per_stock_loaders(
                    X_train, y_train,
                    X_val, y_val,
                    batch_size=batch_size
                )

                model = create_model(
                    model_type=model_type,
                    num_stocks=len(tickers),
                    lookback=lookback,
                    node_feat_dim=len(feature_cols)
                )

                model, history = train_per_stock_torch_model(
                    model=model,
                    train_loader=train_loader,
                    val_loader=val_loader,
                    lr=lr,
                    epochs=epochs,
                    patience=patience
                )

                model.eval()
                with torch.no_grad():
                    x_seq_t = torch.tensor(X_test, dtype=torch.float32, device=DEVICE)
                    pred = model(x_seq_t).cpu().numpy()

            for k, (_, stock_idx) in enumerate(meta_test):
                all_rows.append({
                    "Date": pd.to_datetime(target_date),
                    "Ticker": tickers[stock_idx],
                    "Model": model_type,
                    "Lookback": lookback,
                    "y_true": float(y_test[k, 0]),
                    "y_pred": float(pred[k, 0])
                })

        # =========================
        # HYBRID: multi-stock + node features + graph
        # =========================
        elif model_type == "hybrid":
            train_graph_df = scaled_window_df[scaled_window_df["Date"] <= train_end_date].copy()

            graph_info = build_hybrid_graph(
                train_df=train_graph_df,
                tickers=tickers,
                corr_threshold=CORR_THRESHOLD,
                lift_threshold=LIFT_THRESHOLD,
                min_support=APRIORI_MIN_SUPPORT,
                min_confidence=APRIORI_MIN_CONFIDENCE
            )

            adj_norm = graph_info["norm_adj"]
            adj_tensor = torch.tensor(adj_norm, dtype=torch.float32, device=DEVICE)

            X_seq, X_node, y, target_dates = make_samples_from_tensors(
                close_mat,
                node_feat_tensor,
                dates,
                lookback
            )

            X_seq_train_all = X_seq[:-1]
            X_node_train_all = X_node[:-1]
            y_train_all = y[:-1]

            X_seq_test = X_seq[-1:]
            X_node_test = X_node[-1:]
            y_test = y[-1:]

            (X_seq_train, X_node_train, y_train), (X_seq_val, X_node_val, y_val) = split_train_val_time_order(
                X_seq_train_all, X_node_train_all, y_train_all, val_ratio=0.10
            )

            train_loader, val_loader = make_loaders(
                X_seq_train, X_node_train, y_train,
                X_seq_val, X_node_val, y_val,
                batch_size=batch_size
            )

            model = create_model(
                model_type=model_type,
                num_stocks=len(tickers),
                lookback=lookback,
                node_feat_dim=len(feature_cols)
            )

            model, history = train_torch_model(
                model=model,
                train_loader=train_loader,
                val_loader=val_loader,
                model_type=model_type,
                adj_tensor=adj_tensor,
                lr=lr,
                epochs=epochs,
                patience=patience
            )

            model.eval()
            with torch.no_grad():
                x_seq_t = torch.tensor(X_seq_test, dtype=torch.float32, device=DEVICE)
                x_node_t = torch.tensor(X_node_test, dtype=torch.float32, device=DEVICE)
                pred = model(x_seq_t, x_node_t, adj_tensor).cpu().numpy()

            for i, ticker in enumerate(tickers):
                all_rows.append({
                    "Date": pd.to_datetime(target_date),
                    "Ticker": ticker,
                    "Model": model_type,
                    "Lookback": lookback,
                    "y_true": float(y_test[0, i]),
                    "y_pred": float(pred[0, i])
                })

        else:
            raise ValueError("model_type không hợp lệ")

    result_df = pd.DataFrame(all_rows)
    metrics = regression_metrics(result_df["y_true"].values, result_df["y_pred"].values)

    return result_df, metrics


## 15. Run all five models for a given lookback

In [ ]:
def run_all_models_for_lookback(
    raw_df,
    tickers,
    feature_cols,
    lookback,
    initial_train_days=INITIAL_TRAIN_DAYS,
    test_days=TEST_DAYS,
    lr=LEARNING_RATE,
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    patience=PATIENCE
):
    model_order = ["linear", "dnn", "cnn", "lstm", "hybrid"]

    results = {}
    metrics_rows = []

    for model_type in model_order:
        print("\n" + "#" * 100)
        print(f"RUN MODEL = {model_type.upper()} | LOOKBACK = {lookback}")
        print("#" * 100)

        result_df, metrics = run_expanding_window_backtest(
            raw_df=raw_df,
            tickers=tickers,
            feature_cols=feature_cols,
            lookback=lookback,
            initial_train_days=initial_train_days,
            test_days=test_days,
            model_type=model_type,
            lr=lr,
            epochs=epochs,
            batch_size=batch_size,
            patience=patience
        )

        results[model_type] = {
            "result_df": result_df,
            "metrics": metrics
        }

        metrics_rows.append({
            "Model": model_type.upper(),
            "Lookback": lookback,
            **metrics
        })

    summary_df = pd.DataFrame(metrics_rows).sort_values("MSE").reset_index(drop=True)
    return results, summary_df

## 16. Experiment with lookback = 11

In [ ]:
results_11, summary_11 = run_all_models_for_lookback(
    raw_df=raw_df,
    tickers=TICKERS,
    feature_cols=FEATURE_COLS,
    lookback=11,
    initial_train_days=INITIAL_TRAIN_DAYS,
    test_days=TEST_DAYS,
    lr=LEARNING_RATE,
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    patience=PATIENCE
)

display(summary_11)

## 17. Experiment with lookback = 21

In [ ]:
results_21, summary_21 = run_all_models_for_lookback(
    raw_df=raw_df,
    tickers=TICKERS,
    feature_cols=FEATURE_COLS,
    lookback=21,
    initial_train_days=INITIAL_TRAIN_DAYS,
    test_days=TEST_DAYS,
    lr=LEARNING_RATE,
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    patience=PATIENCE
)

display(summary_21)

## 18. Final comparison across models and lookbacks

In [ ]:
summary_all = pd.concat([summary_11, summary_21], axis=0, ignore_index=True)
summary_all = summary_all.sort_values(["MSE", "RMSE", "MAE"]).reset_index(drop=True)
display(summary_all)

In [ ]:
all_results = pd.concat(
    [x["result_df"] for x in results_11.values()] +
    [x["result_df"] for x in results_21.values()],
    axis=0,
    ignore_index=True
)

display(all_results.head())
print("all_results shape:", all_results.shape)

## 19. Visualization of predictions

In [ ]:
def plot_predictions(result_df, ticker, title_suffix=""):
    sub = result_df[result_df["Ticker"] == ticker].copy().sort_values("Date")

    plt.figure(figsize=(14, 5))
    plt.plot(sub["Date"], sub["y_true"], label="Actual")
    plt.plot(sub["Date"], sub["y_pred"], label="Predicted")
    plt.title(f"{ticker} Actual vs Predicted {title_suffix}")
    plt.xlabel("Date")
    plt.ylabel("Scaled Close")
    plt.legend()
    plt.show()

In [ ]:
plot_predictions(results_11["hybrid"]["result_df"], "AAPL", title_suffix="(HYBRID, Lookback=11)")

In [ ]:
for model_name in ["linear", "dnn", "cnn", "lstm", "hybrid"]:
    plot_predictions(
        results_11[model_name]["result_df"],
        "AAPL",
        title_suffix=f"({model_name.upper()}, Lookback=11)"
    )

In [ ]:
plt.figure(figsize=(10, 5))
plt.bar(summary_all["Model"] + "_L" + summary_all["Lookback"].astype(str), summary_all["MSE"])
plt.xticks(rotation=45)
plt.ylabel("MSE")
plt.title("MSE Comparison Across 5 Models and 2 Lookbacks")
plt.show()

## 20. Save outputs

In [ ]:
summary_all.to_csv("summary_all_models.csv", index=False)
all_results.to_csv("all_results_all_models.csv", index=False)

print("Đã lưu:")
print("- summary_all_models.csv")
print("- all_results_all_models.csv")